# Preprocess TEP Data

Load the interim Parquet files, scale features, and write tabular and time-series datasets to `data/processed/TEP/`. Run `data_conversion.ipynb` first when the interim files are not available.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler


def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "environment.yml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "TEP"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "TEP"
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

if not INTERIM_DATA_PATH.is_dir():
    raise FileNotFoundError(f"Interim data directory does not exist: {INTERIM_DATA_PATH}")

print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [3]:
interim_data_files = sorted(INTERIM_DATA_PATH.glob("*.parquet"))
if not interim_data_files:
    raise FileNotFoundError(f"No .parquet files found in {INTERIM_DATA_PATH}; run data_conversion.ipynb first.")

df_dict = {file_path.stem: pd.read_parquet(file_path) for file_path in interim_data_files}
required_datasets = {
    "TEP_FaultFree_Training",
    "TEP_Faulty_Training",
    "TEP_FaultFree_Testing",
    "TEP_Faulty_Testing",
}
missing_datasets = required_datasets - df_dict.keys()
if missing_datasets:
    missing_names = ", ".join(sorted(missing_datasets))
    raise FileNotFoundError(f"Missing required interim datasets: {missing_names}")

for dataset_name, dataframe in df_dict.items():
    print(f"Loaded {dataset_name}: {dataframe.shape}")

Loaded TEP_FaultFree_Testing: (480000, 55)
Loaded TEP_FaultFree_Training: (250000, 55)
Loaded TEP_Faulty_Testing: (9600000, 55)
Loaded TEP_Faulty_Training: (5000000, 55)


In [4]:
id_cols = ["simulationRun", "sample"]
fault_number_col = "faultNumber"

to_drop_cols = [f"xmeas_{column}" for column in range(23, 42)]
feature_cols = [
    column
    for column in df_dict["TEP_FaultFree_Training"].columns
    if column not in [fault_number_col, *id_cols, *to_drop_cols]
]
if not feature_cols:
    raise ValueError("No feature columns remain after excluding IDs, labels, and differently sampled columns.")

feature_cols_map = {column: f"X{index}" for index, column in enumerate(feature_cols, start=1)}

for required_column in [*id_cols, fault_number_col, *feature_cols]:
    if required_column not in df_dict["TEP_FaultFree_Training"].columns:
        raise KeyError(f"Required column is missing: {required_column}")

df_faultfree_training = df_dict["TEP_FaultFree_Training"]
df_faultfree_training_id = df_faultfree_training[id_cols]
df_faultfree_training_features = df_faultfree_training[feature_cols]
df_faultfree_training_fault = df_faultfree_training[fault_number_col].astype("int")

df_faultfree_training_features_tabular = df_faultfree_training_features.copy()
df_faultfree_training_features_tabular["resample_id"] = np.arange(
    len(df_faultfree_training_features_tabular)
) // 20
df_faultfree_training_features_tabular = (
    df_faultfree_training_features_tabular
    .groupby("resample_id")
    .median()
    .reset_index(drop=True)
)

tabular_scaler = RobustScaler()
scaled_tabular_features = tabular_scaler.fit_transform(df_faultfree_training_features_tabular)
df_faultfree_training_features_tabular = pd.DataFrame(
    scaled_tabular_features,
    columns=feature_cols,
).rename(columns=feature_cols_map)

tabular_output_path = PROCESSED_DATA_PATH / "TEP_Training_FaultFree_Training_Tabular.parquet"
df_faultfree_training_features_tabular.to_parquet(tabular_output_path)
print(f"Wrote {tabular_output_path.name}: {df_faultfree_training_features_tabular.shape}")

Wrote TEP_Training_FaultFree_Training_Tabular.parquet: (12500, 33)


In [5]:
training_scaler = RobustScaler()
scaled_faultfree_training_features = training_scaler.fit_transform(df_faultfree_training_features)
df_faultfree_training_features = pd.DataFrame(
    scaled_faultfree_training_features,
    columns=feature_cols,
    index=df_faultfree_training.index,
)
df_faultfree_training = pd.concat(
    [df_faultfree_training_fault, df_faultfree_training_id, df_faultfree_training_features],
    axis=1,
)

df_faulty_training = df_dict["TEP_Faulty_Training"]
df_faulty_training_features = training_scaler.transform(df_faulty_training[feature_cols])
df_faulty_training_features = pd.DataFrame(
    df_faulty_training_features,
    columns=feature_cols,
    index=df_faulty_training.index,
)
df_faulty_training = pd.concat(
    [
        df_faulty_training[fault_number_col],
        df_faulty_training[id_cols],
        df_faulty_training_features,
    ],
    axis=1,
)

df_training = pd.concat([df_faultfree_training, df_faulty_training], axis=0)
df_training = df_training.astype({column: "int" for column in [*id_cols, fault_number_col]})
df_training = df_training.sort_values(by=[fault_number_col, *id_cols])
df_training = df_training.rename(columns=feature_cols_map)

training_output_path = PROCESSED_DATA_PATH / "TEP_Training.parquet"
df_training.to_parquet(training_output_path)
print(f"Wrote {training_output_path.name}: {df_training.shape}")

Wrote TEP_Training.parquet: (5250000, 36)


In [6]:
df_faultfree_test = df_dict["TEP_FaultFree_Testing"]
df_faulty_test = df_dict["TEP_Faulty_Testing"]

for dataframe_name, dataframe in [
    ("fault-free testing", df_faultfree_test),
    ("faulty testing", df_faulty_test),
]:
    missing_columns = set([*id_cols, fault_number_col, *feature_cols]) - set(dataframe.columns)
    if missing_columns:
        missing_names = ", ".join(sorted(missing_columns))
        raise KeyError(f"Missing columns in {dataframe_name} data: {missing_names}")

df_faultfree_test_features = training_scaler.transform(df_faultfree_test[feature_cols])
df_faultfree_test_features = pd.DataFrame(
    df_faultfree_test_features,
    columns=feature_cols,
    index=df_faultfree_test.index,
)
df_faultfree_test = pd.concat(
    [
        df_faultfree_test[fault_number_col],
        df_faultfree_test[id_cols],
        df_faultfree_test_features,
    ],
    axis=1,
)

df_faulty_test_features = training_scaler.transform(df_faulty_test[feature_cols])
df_faulty_test_features = pd.DataFrame(
    df_faulty_test_features,
    columns=feature_cols,
    index=df_faulty_test.index,
)
df_faulty_test = pd.concat(
    [
        df_faulty_test[fault_number_col],
        df_faulty_test[id_cols],
        df_faulty_test_features,
    ],
    axis=1,
)

df_test = pd.concat([df_faultfree_test, df_faulty_test], axis=0)
df_test = df_test.astype({column: "int" for column in [*id_cols, fault_number_col]})
df_test = df_test.sort_values(by=[fault_number_col, *id_cols])
df_test = df_test.rename(columns=feature_cols_map)

testing_output_path = PROCESSED_DATA_PATH / "TEP_Testing.parquet"
df_test.to_parquet(testing_output_path)
print(f"Wrote {testing_output_path.name}: {df_test.shape}")

Wrote TEP_Testing.parquet: (10080000, 36)
